# Evaluación Parcial N°2: QuickTrack
Integración con Python (pymongo) y Azure Cosmos DB for MongoDB.

BDY0101 Base de Datos No Relacionales, sección 002D. Grupo 07: Anyara Rosso e Italo Zapata.


In [1]:
!pip install pymongo


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 2.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pymongo]


In [2]:
import sys
import logging
from pymongo import MongoClient
from pymongo.errors import PyMongoError, DuplicateKeyError
from google.colab import userdata

# El log se muestra en pantalla y también se guarda en el archivo quicktrack.log
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[logging.FileHandler("quicktrack.log"), logging.StreamHandler(sys.stdout)],
    force=True
)

# Solo se registran los mensajes de nuestro programa
logging.getLogger("pymongo").setLevel(logging.WARNING)


In [3]:
# La cadena de conexión está guardada en los Secrets de Colab
uri = userdata.get("COSMOS_URI")

try:
    client = MongoClient(uri)
    client.admin.command("ping")
    db = client["quick-track-company"]
    logging.info("Conexión exitosa con Azure Cosmos DB")
    for nombre in sorted(db.list_collection_names()):
        print(nombre, ":", db[nombre].count_documents({}), "documentos")
except PyMongoError as e:
    logging.error(f"No se pudo conectar: {e}")


/tmp/ipykernel_10158/1381193139.py:5: UserWarning: You appear to be connected to a CosmosDB cluster. For more information regarding feature compatibility and support please visit https://www.mongodb.com/supportability/cosmosdb
  client = MongoClient(uri)


2026-11-05 20:57:20,103 | INFO | Conexión exitosa con Azure Cosmos DB
clientes : 101 documentos
ordenes-entrega : 100 documentos
rutas : 100 documentos
seguimientos : 100 documentos
valoraciones : 100 documentos


In [4]:
ordenes = db["ordenes-entrega"]
seguimientos = db["seguimientos"]
clientes = db["clientes"]

ESTADOS = ["En preparación", "Despachado", "En tránsito", "Entregado", "Cancelado"]
MEDIOS_PAGO = ["Efectivo", "Tarjeta", "Transferencia"]

def validar_orden(orden):
    campos = ["_id", "cliente_id", "ruta_id", "fecha_envio", "estado", "medio_pago", "total"]
    for campo in campos:
        if campo not in orden:
            raise ValueError(f"falta el campo {campo}")
    if orden["estado"] not in ESTADOS:
        raise ValueError(f"estado no válido: {orden['estado']}")
    if orden["medio_pago"] not in MEDIOS_PAGO:
        raise ValueError(f"medio de pago no válido: {orden['medio_pago']}")
    if orden["total"] <= 0:
        raise ValueError("el total debe ser mayor a 0")
    if clientes.count_documents({"_id": orden["cliente_id"]}) == 0:
        raise ValueError(f"no existe el cliente {orden['cliente_id']}")


In [5]:
def crear_orden(orden):
    try:
        validar_orden(orden)
        ordenes.insert_one(orden)
        # Cada orden nueva parte con su seguimiento
        seguimientos.insert_one({
            "_id": orden["_id"].replace("orden_", "seg_"),
            "orden_id": orden["_id"],
            "estado_actual": orden["estado"],
            "historial": [{"estado": orden["estado"], "fecha": orden["fecha_envio"]}]
        })
        logging.info(f"Orden creada: {orden['_id']}")
    except ValueError as e:
        logging.warning(f"Orden {orden['_id']} rechazada: {e}")
    except DuplicateKeyError:
        logging.error(f"La orden {orden['_id']} ya existe")
    except PyMongoError as e:
        logging.error(f"Error de base de datos: {e}")


def buscar_ordenes(filtro):
    try:
        resultado = list(ordenes.find(filtro, {"cliente_id": 1, "estado": 1, "total": 1}))
        logging.info(f"Búsqueda {filtro}, órdenes encontradas: {len(resultado)}")
        return resultado
    except PyMongoError as e:
        logging.error(f"Error al buscar: {e}")
        return []


def actualizar_estado(orden_id, nuevo_estado, fecha):
    if nuevo_estado not in ESTADOS:
        logging.warning(f"Estado no válido: {nuevo_estado}")
        return
    try:
        resultado = ordenes.update_one({"_id": orden_id}, {"$set": {"estado": nuevo_estado}})
        if resultado.matched_count == 0:
            logging.warning(f"No existe la orden {orden_id}")
            return
        # El seguimiento se actualiza junto con la orden
        seguimientos.update_one(
            {"orden_id": orden_id},
            {
                "$set": {"estado_actual": nuevo_estado},
                "$push": {"historial": {"estado": nuevo_estado, "fecha": fecha}}
            }
        )
        logging.info(f"Orden {orden_id} actualizada a {nuevo_estado}")
    except PyMongoError as e:
        logging.error(f"Error al actualizar: {e}")


def eliminar_orden(orden_id):
    try:
        resultado = ordenes.delete_one({"_id": orden_id})
        if resultado.deleted_count == 0:
            logging.warning(f"No existe la orden {orden_id}")
            return
        seguimientos.delete_one({"orden_id": orden_id})
        logging.info(f"Orden {orden_id} eliminada junto con su seguimiento")
    except PyMongoError as e:
        logging.error(f"Error al eliminar: {e}")


In [6]:
nueva_orden = {
    "_id": "orden_102",
    "cliente_id": 101,
    "ruta_id": "ruta_25",
    "fecha_envio": "2026-11-06T00:00:00",
    "estado": "En preparación",
    "medio_pago": "Transferencia",
    "total": 64990
}

crear_orden(nueva_orden)

for orden in buscar_ordenes({"cliente_id": 101}):
    print(orden)

actualizar_estado("orden_102", "Despachado", "2026-11-07T00:00:00")
print(seguimientos.find_one({"orden_id": "orden_102"}))


2026-11-05 21:03:43,050 | INFO | Orden creada: orden_102
2026-11-05 21:03:43,088 | INFO | Búsqueda {'cliente_id': 101}, órdenes encontradas: 1
{'_id': 'orden_102', 'cliente_id': 101, 'estado': 'En preparación', 'total': 64990}
2026-11-05 21:03:43,184 | INFO | Orden orden_102 actualizada a Despachado
{'_id': 'seg_102', 'orden_id': 'orden_102', 'estado_actual': 'Despachado', 'historial': [{'estado': 'En preparación', 'fecha': '2026-11-06T00:00:00'}, {'estado': 'Despachado', 'fecha': '2026-11-07T00:00:00'}]}


In [7]:
# Orden repetida
crear_orden(nueva_orden)

# Falta un campo
crear_orden({"_id": "orden_103", "cliente_id": 101, "total": 20000})

# Total negativo
crear_orden({
    "_id": "orden_103",
    "cliente_id": 101,
    "ruta_id": "ruta_25",
    "fecha_envio": "2026-11-06T00:00:00",
    "estado": "En preparación",
    "medio_pago": "Efectivo",
    "total": -5000
})

# Cliente que no existe
crear_orden({
    "_id": "orden_103",
    "cliente_id": 999,
    "ruta_id": "ruta_25",
    "fecha_envio": "2026-11-06T00:00:00",
    "estado": "En preparación",
    "medio_pago": "Efectivo",
    "total": 20000
})

# Estado que no existe
actualizar_estado("orden_102", "Perdido", "2026-11-07T00:00:00")

# Orden que no existe
actualizar_estado("orden_999", "Entregado", "2026-11-07T00:00:00")


2026-11-05 21:03:52,242 | ERROR | La orden orden_102 ya existe
2026-11-05 21:03:52,244 | WARNING | Orden orden_103 rechazada: falta el campo ruta_id
2026-11-05 21:03:52,246 | WARNING | Orden orden_103 rechazada: el total debe ser mayor a 0
2026-11-05 21:03:52,295 | WARNING | Orden orden_103 rechazada: no existe el cliente 999
2026-11-05 21:03:52,296 | WARNING | Estado no válido: Perdido
2026-11-05 21:03:52,327 | WARNING | No existe la orden orden_999


In [8]:
eliminar_orden("orden_102")

# Si se intenta borrar otra vez, la orden ya no existe
eliminar_orden("orden_102")

print("Órdenes:", ordenes.count_documents({}))
print("Seguimientos de orden_102:", seguimientos.count_documents({"orden_id": "orden_102"}))


2026-11-05 21:04:00,098 | INFO | Orden orden_102 eliminada junto con su seguimiento
2026-11-05 21:04:00,128 | WARNING | No existe la orden orden_102
Órdenes: 100
Seguimientos de orden_102: 0


In [9]:
!cat quicktrack.log


2026-11-05 20:57:20,103 | INFO | Conexión exitosa con Azure Cosmos DB
2026-11-05 21:03:43,050 | INFO | Orden creada: orden_102
2026-11-05 21:03:43,088 | INFO | Búsqueda {'cliente_id': 101}, órdenes encontradas: 1
2026-11-05 21:03:43,184 | INFO | Orden orden_102 actualizada a Despachado
2026-11-05 21:03:52,242 | ERROR | La orden orden_102 ya existe
2026-11-05 21:03:52,244 | WARNING | Orden orden_103 rechazada: falta el campo ruta_id
2026-11-05 21:03:52,246 | WARNING | Orden orden_103 rechazada: el total debe ser mayor a 0
2026-11-05 21:03:52,295 | WARNING | Orden orden_103 rechazada: no existe el cliente 999
2026-11-05 21:03:52,296 | WARNING | Estado no válido: Perdido
2026-11-05 21:03:52,327 | WARNING | No existe la orden orden_999
2026-11-05 21:04:00,098 | INFO | Orden orden_102 eliminada junto con su seguimiento
2026-11-05 21:04:00,128 | WARNING | No existe la orden orden_102


In [10]:
client.close()
logging.info("Conexión cerrada")


2026-11-05 21:04:19,613 | INFO | Conexión cerrada
